# TalentScout — Ingestion Pipeline

Notebook ini menjalankan ingestion korpus resume ke Qdrant:
**clean → chunk → filter → embed → upsert**, ditambah summary per-resume di collection terpisah.

## Resume-from-crash mechanism

- Setiap chunk dapat **deterministic point ID** = `uuid5(NAMESPACE, f"{resume_id}_{chunk_index}")`. Re-upsert ID sama = overwrite, bukan duplicate.
- Tiap resume di-upsert dalam **satu `client.upsert()` call** → Qdrant menjamin atomic per-request. Tidak ada state "resume setengah jadi".
- Sebelum proses, scroll collection → kumpulkan `done_resume_ids`. Skip yang sudah ada. Pending = `all_ids - done_ids`.
- Cell apapun bisa di-rerun tanpa duplikasi data. Untuk full reset, pakai cell **destructive** di akhir notebook.

## Source of truth

Qdrant sendiri. Tidak ada checkpoint file lokal yang bisa drift dari realitas database.

## 1. Setup

Resolusi project root, secrets loading, dan import config.

In [10]:
import sys
import os
import tomllib
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

secrets_path = PROJECT_ROOT / ".streamlit" / "secrets.toml"
with open(secrets_path, "rb") as f:
    _secrets = tomllib.load(f)
for k, v in _secrets.items():
    if isinstance(v, str):
        os.environ.setdefault(k, v)

from src import config

print(f"Project root : {PROJECT_ROOT}")
print(f"Embedding    : {config.EMBEDDING_MODEL}")
print(f"Chat model   : {config.CHAT_MODEL}")
print(f"Collections  : {config.CHUNK_COLLECTION}, {config.SUMMARY_COLLECTION}")
print(f"Categories   : {len(config.CATEGORIES)}")

Project root : /home/yusuf/Documents/purwadhika/module_3/capstone_3_TalentScout
Embedding    : text-embedding-3-small
Chat model   : gpt-4o-mini
Collections  : resume_chunks, resume_summaries
Categories   : 24


In [11]:
import pandas as pd

CSV_PATH = PROJECT_ROOT / "dataset" / "resume" / "Resume.csv"
df_raw = pd.read_csv(CSV_PATH)

print(f"Rows         : {len(df_raw)}")
print(f"Columns      : {list(df_raw.columns)}")
print(f"Unique cats  : {df_raw['Category'].nunique()}")
print(f"Total nulls  : {df_raw.isna().sum().sum()}")

Rows         : 2484
Columns      : ['ID', 'Resume_str', 'Resume_html', 'Category']
Unique cats  : 24
Total nulls  : 0


## 2. Cleaning pipeline

Tiga langkah berurutan:

1. **Dedup HTML** — `drop_duplicates(subset='Resume_html', keep='first')`. 4 rows di 2 cluster byte-identik. Net: drop 2.
2. **Length filter** — drop `Resume_str` dengan panjang < 500 char. Membuang 1 row 21-char garbage.
3. **Normalize** — `html.unescape` + NFKC + exotic whitespace (`\xa0`, figure space, narrow nbsp, `\t`) → space biasa. **JANGAN** collapse internal `\s{3,}` (delimiter section yang dipakai chunker). **JANGAN** lowercase.

Expected counts: `2484 → 2482 → 2481`.

In [12]:
import html
import unicodedata

EXOTIC_WS = "\xa0\u2007\u202f\t"  # nbsp, figure space, narrow nbsp, tab


def clean_resume(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    for ch in EXOTIC_WS:
        text = text.replace(ch, " ")
    return text.strip()


n0 = len(df_raw)
df = df_raw.drop_duplicates(subset="Resume_html", keep="first").copy()
n1 = len(df)

df = df.loc[df["Resume_str"].fillna("").str.len() >= 500].copy()
n2 = len(df)

df["Resume_clean"] = df["Resume_str"].map(clean_resume)

print(f"Dedup HTML      : {n0} → {n1}  (dropped {n0 - n1})")
print(f"Length ≥500     : {n1} → {n2}  (dropped {n1 - n2})")
print(f"Final corpus    : {n2} resumes")
print(f"Median length   : {df['Resume_clean'].str.len().median():.0f} char")

Dedup HTML      : 2484 → 2482  (dropped 2)
Length ≥500     : 2482 → 2481  (dropped 1)
Final corpus    : 2481 resumes
Median length   : 5875 char


## 3. Chunker

Hybrid section-aware chunker. Logic dikunci di fase eksplorasi — di-port verbatim.

**Algoritma:**

1. Cari semua posisi header section via `SYNONYM_MAP` regex (`\s{3,}(synonym)\s{3,}`, case-insensitive). Synonim diurutkan panjang-desc agar `Education and Training` match sebelum `Education`.
2. **Collapse** marker berurutan dengan canonical sama (mis. "Experience" lalu "Work History" beruntun) → buang yang redundan; section pertama membentang sampai canonical berbeda berikutnya.
3. Kalau **< 3 unique canonical section** terdeteksi → fallback: sliding window seragam atas seluruh teks, semua chunk tag `body`.
4. Kalau **≥ 3** → section-aware:
   - Preamble (teks sebelum header pertama) → tag `title`.
   - Tiap section: konten mulai **setelah** kata header, berakhir di header berikut (atau akhir teks).
   - Section > 500 char → sub-split via sliding window (size 500, overlap 50).
5. Kata header di-strip dari konten chunk (konsistensi minimalist).

Output tiap chunk: dict `{resume_id, category, section, chunk_index, content}`.

In [13]:
import re

SYNONYM_MAP = {
    'summary'                : ['Summary', 'Professional Summary', 'Career Overview', 'Professional Overview', 'Profile', 'Objective'],
    'skills'                 : ['Skills', 'Core Qualifications', 'Skill Highlights', 'Technical Skills', 'Core Competencies', 'Key Skills'],
    'experience'             : ['Experience', 'Professional Experience', 'Work History', 'Work Experience', 'Employment History'],
    'education'              : ['Education', 'Education and Training', 'Academic Background'],
    'highlights'             : ['Highlights', 'Career Highlights'],
    'accomplishments'        : ['Accomplishments', 'Core Accomplishments', 'Achievements'],
    'certifications'         : ['Certifications', 'Certificates', 'Professional Certifications', 'Licenses'],
    'additional_information' : ['Additional Information'],
    'languages'              : ['Languages'],
}


def chunk_resume(resume_id, category, text, synonym_map,
                 chunk_size=500, overlap=50, min_sections=3):
    """Hybrid section-aware chunker with sliding-window sub-chunks."""

    # 1. Find all section markers. Longer synonyms tried first.
    markers = []
    for canonical, synonyms in synonym_map.items():
        syn_sorted = sorted(synonyms, key=len, reverse=True)
        pat = re.compile(
            r'\s{3,}(' + '|'.join(re.escape(s) for s in syn_sorted) + r')\s{3,}',
            re.IGNORECASE,
        )
        for m in pat.finditer(text):
            markers.append((m.start(1), m.end(1), canonical))

    markers.sort()

    # 1b. Collapse consecutive same-canonical markers so the first marker's
    # content spans through to the next *different* canonical section.
    collapsed = []
    for m in markers:
        if collapsed and collapsed[-1][2] == m[2]:
            continue
        collapsed.append(m)
    markers = collapsed

    # 2. Decide path.
    unique_sections = {name for _, _, name in markers}
    if len(unique_sections) < min_sections:
        return _sliding(resume_id, category, 'body', text.strip(),
                        chunk_size, overlap, base_idx=0)

    # 3. Section-aware path.
    chunks = []

    # Preamble before first header -> 'title'.
    if markers[0][0] > 0:
        preamble = text[:markers[0][0]].strip()
        if preamble:
            chunks.extend(_sliding(resume_id, category, 'title', preamble,
                                   chunk_size, overlap, base_idx=len(chunks)))

    # Each detected section.
    for i, (start, end, name) in enumerate(markers):
        content_start = end
        content_end = markers[i + 1][0] if i + 1 < len(markers) else len(text)
        content = text[content_start:content_end].strip()
        if content:
            chunks.extend(_sliding(resume_id, category, name, content,
                                   chunk_size, overlap, base_idx=len(chunks)))
    return chunks


def _sliding(resume_id, category, section, text,
             chunk_size, overlap, base_idx):
    """Sub-split text via sliding window. Stops when end is reached."""
    if not text:
        return []
    if len(text) <= chunk_size:
        return [{
            'resume_id': resume_id, 'category': category, 'section': section,
            'chunk_index': base_idx, 'content': text,
        }]
    chunks = []
    step = chunk_size - overlap
    pos = 0
    while pos < len(text):
        sub = text[pos:pos + chunk_size]
        chunks.append({
            'resume_id': resume_id, 'category': category, 'section': section,
            'chunk_index': base_idx + len(chunks), 'content': sub,
        })
        if pos + chunk_size >= len(text):
            break
        pos += step
    return chunks

In [14]:
all_chunks = []
for _, row in df.iterrows():
    all_chunks.extend(
        chunk_resume(row["ID"], row["Category"], row["Resume_clean"], SYNONYM_MAP)
    )

chunk_lens = pd.Series([len(c["content"]) for c in all_chunks])

print(f"Resumes processed : {len(df)}")
print(f"Total chunks      : {len(all_chunks):,}")
print(f"Chunks per resume : {len(all_chunks) / len(df):.1f} avg")
print()
print("Chunk length distribution:")
print(chunk_lens.describe(percentiles=[0.01, 0.5, 0.95, 0.99]).round(0).to_string())
print()
print("Chunks per section:")
print(pd.Series([c["section"] for c in all_chunks]).value_counts().to_string())

Resumes processed : 2481
Total chunks      : 41,742
Chunks per resume : 16.8 avg

Chunk length distribution:
count    41742.0
mean       397.0
std        161.0
min          1.0
1%          13.0
50%        500.0
95%        500.0
99%        500.0
max        500.0

Chunks per section:
experience                23696
skills                     4151
education                  3649
summary                    3168
title                      2752
accomplishments            1614
highlights                 1116
additional_information      635
certifications              392
body                        295
languages                   274


## 4. Chunk filter

Dijalankan **antara chunk generation dan embedding** — hemat biaya embed chunk yang akan dibuang.

Empat aturan (dievaluasi berurutan, aturan pertama yang cocok = drop):

1. **Title = restatement kategori** — `title` section yang isinya (uppercased, hyphen→space dinormalkan) persis salah satu dari 24 nama kategori, mis. "SALES" di resume kategori SALES, atau "BUSINESS DEVELOPMENT". Ini menargetkan *akar* (restatement label) bukan *gejala* (panjang). Menggantikan proxy `<20` yang terbukti menyeret 751 judul jabatan asli ("GRAPHIC DESIGNER", "HR MANAGER", dst.).
2. **Header word leak** — konten yang persis = nama section canonical/synonim (artefak pasca-collapse, mis. konten literal "Skills"). Bandingkan `lower()` agar tahan variasi kapitalisasi.
3. **PII placeholder** — string template seperti "Company Name", "City", "Total Years    Last Used". Bandingkan **case-sensitive** (sengaja: artefak mesin sensor selalu format kanonik; case-fold = risiko salah-tangkap konten valid).
4. **`< 3` char** — lantai universal: page number, sisa tanda baca, single char.

**Sengaja dipertahankan** (akan salah-drop kalau pakai gut-threshold panjang): kode sertifikasi 3-char (`CPA`, `PMP`, `AIT`), bahasa single-word (`Spanish`), keyword pendek (`AWS`, `PHP`, `SQL`), dan judul jabatan pendek (`HR MANAGER`, `GRAPHIC DESIGNER`).

In [15]:
CATEGORY_RESTATEMENTS = (
    {c.upper() for c in config.CATEGORIES}
    | {c.upper().replace("-", " ") for c in config.CATEGORIES}
)

CANONICAL_HEADER_WORDS = {
    'skills', 'experience', 'education', 'summary',
    'highlights', 'accomplishments', 'certifications',
    'additional information', 'languages',
    # synonyms that may leak as content after marker collapse
    'professional summary', 'professional experience', 'work history',
    'core qualifications', 'skill highlights', 'core accomplishments',
    'career overview', 'professional overview',
}

PII_PLACEHOLDERS = {
    'Company Name', 'City', 'State', 'USA',
    'Total Years    Last Used',
}


def is_keepable(chunk):
    content = chunk['content'].strip()
    # 1. Title chunk that is just a category-name restatement.
    if chunk['section'] == 'title' and content.upper() in CATEGORY_RESTATEMENTS:
        return False
    # 2. Header word leaked as content (post-collapse artifact).
    if content.lower() in CANONICAL_HEADER_WORDS:
        return False
    # 3. Known PII placeholder.
    if content in PII_PLACEHOLDERS:
        return False
    # 4. Hard floor — 1-2 char content is noise.
    if len(content) < 3:
        return False
    return True


def _drop_reason(chunk):
    """First rule that fires — mirrors is_keepable order for an honest audit tally."""
    content = chunk['content'].strip()
    if chunk['section'] == 'title' and content.upper() in CATEGORY_RESTATEMENTS:
        return 'title = category restatement'
    if content.lower() in CANONICAL_HEADER_WORDS:
        return 'header word leak'
    if content in PII_PLACEHOLDERS:
        return 'PII placeholder'
    if len(content) < 3:
        return '<3 chars'
    return None


kept_chunks = [c for c in all_chunks if is_keepable(c)]
dropped = len(all_chunks) - len(kept_chunks)

reasons = pd.Series(
    [r for c in all_chunks if (r := _drop_reason(c)) is not None]
)

print(f"Before filter : {len(all_chunks):,}")
print(f"After filter  : {len(kept_chunks):,}")
print(f"Dropped       : {dropped:,}  ({dropped / len(all_chunks) * 100:.2f}%)")
print()
print("Drop breakdown (first rule that fires):")
print(reasons.value_counts().to_string())

Before filter : 41,742
After filter  : 41,497
Dropped       : 245  (0.59%)

Drop breakdown (first rule that fires):
title = category restatement    215
PII placeholder                  16
header word leak                 12
<3 chars                          2


## 5. Sanity inspection

Gate manual sebelum embedding (langkah mahal). Ambil 3 chunk acak per section dari `kept_chunks`, eyeball: konten match label section? Ada noise yang lolos filter? Judul jabatan pendek benar terselamatkan di section `title`?

In [16]:
import random

rng = random.Random(42)
by_section = {}
for c in kept_chunks:
    by_section.setdefault(c["section"], []).append(c)

print(f"Total kept: {len(kept_chunks):,} across {len(by_section)} sections")
print()
for section in sorted(by_section):
    bucket = by_section[section]
    sample = rng.sample(bucket, min(3, len(bucket)))
    print(f"=== {section}  (n={len(bucket):,}) ===")
    for c in sample:
        preview = " ".join(c["content"][:160].split())
        print(f"  id={c['resume_id']} | {c['category']} | idx={c['chunk_index']} | {len(c['content'])} char")
        print(f"    {preview!r}")
    print()

Total kept: 41,497 across 11 sections

=== accomplishments  (n=1,614) ===
  id=28949406 | CONSTRUCTION | idx=15 | 365 char
    'I have received "Exceeds Expectations" on my end of year performance reviews with Ginkgo Residential and Fairfield Residential. Volunteering Volunte'
  id=12334140 | INFORMATION-TECHNOLOGY | idx=15 | 500 char
    'a different word processing program, the process is similar. Select the content of your cover letter then choose a font and a font size. You may need to try a'
  id=21830565 | HR | idx=3 | 418 char
    'Established HR infrastructure as companies transitioned from small to mid-size. Trained HR staff in proper creation and retention of HR documentation Institut'

=== additional_information  (n=632) ===
  id=29002596 | DIGITAL-MEDIA | idx=20 | 148 char
    'ADDITIONAL INFORMATION Fairport Soccer Club - Team Manager Fairport High School Girls Soccer and Girls Lacrosse - Booster club member Skills'
  id=17025292 | CONSULTANT | idx=14 | 40 char
    'https:

## 6. Embed + upsert (resume-safe)

Loop inti ingestion. Arsitektur sesuai keputusan terkunci:

- **Deterministic point ID** = `uuid5(INGESTION_NAMESPACE, f"{resume_id}_{chunk_index}")`. Re-upsert ID sama = overwrite. `INGESTION_NAMESPACE` diturunkan dari **seed string tetap** (bukan UUID literal) → reproducible walau notebook ditulis ulang.
- **Per-resume atomic upsert**: semua chunk 1 resume dalam 1 `upsert()` call. Qdrant atomic per-request → tidak ada partial-resume.
- **Embed batched lintas resume** (50/call) untuk efisiensi; commit ke Qdrant tetap per-resume.
- **Resume-from-crash**: scroll collection → `done_ids`; proses hanya `all − done`. Source of truth = Qdrant, tidak ada file lokal.
- **Reset**: cell destruktif `delete_collection` (akan ditaruh di akhir notebook).

Cell di bawah: init client + namespace + ensure collection (create-if-not-exists, payload index `category` — satu-satunya hard filter).

In [17]:
import uuid
from openai import OpenAI
from qdrant_client import QdrantClient

# Namespace derived from a fixed seed string (NOT a stored magic UUID literal).
# Deterministic + reproducible across notebook rewrites. Changing the seed
# string = a brand-new ID space (would orphan all existing points).
INGESTION_NAMESPACE = uuid.uuid5(uuid.NAMESPACE_DNS, "talentscout-ingestion-v1")

openai_client = OpenAI(api_key=config.OPENAI_API_KEY)
qdrant = QdrantClient(
    url=config.QDRANT_URL,
    api_key=config.QDRANT_API_KEY,
    timeout=60,
)

_existing = [c.name for c in qdrant.get_collections().collections]
print(f"Namespace : {INGESTION_NAMESPACE}")
print(f"Qdrant up : {len(_existing)} collections — {_existing}")

Namespace : 004a3248-0727-55de-b203-571698837fd3
Qdrant up : 2 collections — ['resume_summaries', 'resume_chunks']


In [ ]:
from qdrant_client.models import Distance, VectorParams, PayloadSchemaType

EMBED_DIM = 1536  # text-embedding-3-small

# Option A: every hard filter lives nested under metadata.*. Two indexes are
# required end-state: category (KEYWORD, the only user-facing hard filter) and
# resume_id (INTEGER, used by get_full_resume / get_summary scroll-by-id).
REQUIRED_INDEXES = {
    "metadata.category": PayloadSchemaType.KEYWORD,
    "metadata.resume_id": PayloadSchemaType.INTEGER,
}


def ensure_collection(name, vector_size=EMBED_DIM):
    """Idempotent on the END STATE: the collection AND every required
    payload index exist. Re-run safe.

    Indexes are ensured even when the collection already exists — a
    pre-existing collection created without one would otherwise silently
    400 on every filtered query against that field (this is what the
    validation gates caught, twice: metadata.category, then later
    metadata.resume_id). 'ensure' must mean end-state, not
    create-if-absent."""
    existing = {c.name for c in qdrant.get_collections().collections}
    if name not in existing:
        qdrant.create_collection(
            collection_name=name,
            vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
        )
        print(f"[{name}] created — size={vector_size}, cosine")
    else:
        info = qdrant.get_collection(name)
        print(f"[{name}] exists — {info.points_count:,} points, reuse")

    # Ensure each required index regardless of create/reuse. langchain-qdrant
    # filters on metadata.<key>, so indexes MUST be the nested keys.
    schema = getattr(qdrant.get_collection(name), "payload_schema", None) or {}
    for field, kind in REQUIRED_INDEXES.items():
        if field in schema:
            print(f"  [{name}] {field} index present")
        else:
            qdrant.create_payload_index(
                collection_name=name,
                field_name=field,
                field_schema=kind,
            )
            print(f"  [{name}] {field} index created ({kind})")


ensure_collection(config.CHUNK_COLLECTION)
ensure_collection(config.SUMMARY_COLLECTION)

In [33]:
import time
from openai import (
    RateLimitError, APITimeoutError, APIConnectionError, InternalServerError,
)

EMBED_BATCH = 50  # chunks per OpenAI embedding call

# Transient errors worth retrying. BadRequest/auth are NOT here — those are
# bugs, not blips, and must fail loud immediately.
_RETRYABLE = (RateLimitError, APITimeoutError, APIConnectionError, InternalServerError)


def embed_texts(texts):
    """Embed strings via OpenAI, batched at EMBED_BATCH. Hand-rolled
    exponential backoff on transient errors (no extra dependency). Fail-loud
    after retries exhausted — the resume mechanism makes a hard stop safe:
    just re-run the cell, scroll skips what is already done."""
    out = []
    for start in range(0, len(texts), EMBED_BATCH):
        batch = texts[start:start + EMBED_BATCH]
        resp = None
        for attempt in range(5):
            try:
                resp = openai_client.embeddings.create(
                    model=config.EMBEDDING_MODEL, input=batch
                )
                break
            except _RETRYABLE as e:
                if attempt == 4:
                    raise
                wait = 2 ** attempt
                print(f"  transient embed error ({type(e).__name__}); "
                      f"retry {attempt + 1}/4 in {wait}s")
                time.sleep(wait)
        # API may return out of order; .index re-aligns to input order.
        out.extend(d.embedding for d in sorted(resp.data, key=lambda d: d.index))
    return out


def scroll_done_resume_ids(collection):
    """Resume-from-crash source of truth: set of resume_ids already present in
    `collection`. Option A nests it at payload["metadata"]["resume_id"], so we
    fetch the small metadata dict (vectors excluded to save bandwidth) and
    paginate the scroll cursor until exhausted."""
    done = set()
    offset = None
    while True:
        points, offset = qdrant.scroll(
            collection_name=collection,
            with_payload=["metadata"],
            with_vectors=False,
            limit=10_000,
            offset=offset,
        )
        for pt in points:
            md = pt.payload.get("metadata") or {}
            if "resume_id" in md:
                done.add(md["resume_id"])
        if offset is None:
            break
    return done


print("Helpers ready: embed_texts(), scroll_done_resume_ids()")

Helpers ready: embed_texts(), scroll_done_resume_ids()


In [24]:
import time
import uuid
from qdrant_client.models import PointStruct

CHUNK_COL = config.CHUNK_COLLECTION
RESUME_BATCH = 50  # resumes embedded together per OpenAI batch window

# 1. Group kept chunks by resume_id — resume is the atomic commit unit.
by_resume = {}
for c in kept_chunks:
    by_resume.setdefault(c["resume_id"], []).append(c)

# 2. Resume-from-crash: skip resumes already fully in Qdrant.
done = scroll_done_resume_ids(CHUNK_COL)
pending = [rid for rid in by_resume if rid not in done]
print(f"Resumes total : {len(by_resume):,}")
print(f"Already done  : {len(done):,}")
print(f"Pending       : {len(pending):,}")

# 3. Embed batched across a window of resumes; upsert ATOMIC per resume.
t0 = time.time()
processed = 0
for i in range(0, len(pending), RESUME_BATCH):
    window = pending[i:i + RESUME_BATCH]
    window_chunks = [c for rid in window for c in by_resume[rid]]

    vectors = embed_texts([c["content"] for c in window_chunks])

    cursor = 0
    for rid in window:
        points = []
        for c in by_resume[rid]:
            v = vectors[cursor]
            cursor += 1
            pid = str(uuid.uuid5(
                INGESTION_NAMESPACE, f"{c['resume_id']}_{c['chunk_index']}"
            ))
            points.append(PointStruct(
                id=pid,
                vector=v,
                payload={
                    "page_content": c["content"],
                    "metadata": {
                        "resume_id": c["resume_id"],
                        "category": c["category"],
                        "section": c["section"],
                        "chunk_index": c["chunk_index"],
                    },
                },
            ))
        # One HTTP call per resume -> Qdrant atomic per request.
        qdrant.upsert(collection_name=CHUNK_COL, points=points)
        processed += 1

    el = time.time() - t0
    rate = processed / el if el else 0
    eta = (len(pending) - processed) / rate if rate else 0
    print(f"  {processed:,}/{len(pending):,} | {el:,.0f}s | ~{eta:,.0f}s left")

print(f"Chunks ingested: {processed:,} resumes in {time.time() - t0:,.0f}s.")

Resumes total : 2,481
Already done  : 0
Pending       : 2,481
  50/2,481 | 55s | ~2,660s left
  100/2,481 | 101s | ~2,414s left
  150/2,481 | 138s | ~2,150s left
  200/2,481 | 176s | ~2,005s left
  250/2,481 | 220s | ~1,961s left
  300/2,481 | 263s | ~1,912s left
  350/2,481 | 306s | ~1,862s left
  400/2,481 | 343s | ~1,784s left
  450/2,481 | 388s | ~1,752s left
  500/2,481 | 430s | ~1,706s left
  550/2,481 | 470s | ~1,649s left
  600/2,481 | 512s | ~1,604s left
  650/2,481 | 552s | ~1,556s left
  700/2,481 | 597s | ~1,519s left
  750/2,481 | 646s | ~1,492s left
  800/2,481 | 689s | ~1,448s left
  850/2,481 | 735s | ~1,410s left
  900/2,481 | 778s | ~1,367s left
  950/2,481 | 819s | ~1,320s left
  1,000/2,481 | 863s | ~1,278s left
  1,050/2,481 | 896s | ~1,222s left
  1,100/2,481 | 935s | ~1,174s left
  1,150/2,481 | 978s | ~1,132s left
  1,200/2,481 | 1,023s | ~1,092s left
  1,250/2,481 | 1,067s | ~1,050s left
  1,300/2,481 | 1,108s | ~1,006s left
  1,350/2,481 | 1,146s | ~960s left


In [25]:
ci = qdrant.get_collection(config.CHUNK_COLLECTION)
print("points_count :", ci.points_count, " (target ≈ 41,497)")
print("resume done  :", len(scroll_done_resume_ids(config.CHUNK_COLLECTION)), " (target 2,481)")


points_count : 41497  (target ≈ 41,497)
resume done  : 2481  (target 2,481)


## 7. Summary per-resume (gerbang triase)

Collection kedua, `resume_summaries`, melayani peran yang tidak bisa dilayani `resume_chunks`: **recall di level kandidat**, bukan level potongan teks.

Arsitektur *coarse-to-fine*: pencarian luas dulu di `resume_summaries` (satu vektor = satu kandidat) untuk menyaring siapa yang relevan, lalu *zoom-in* ke `resume_chunks` via `resume_id` untuk detail. Kardinalitas **1 point : 1 kandidat** juga membuat agregasi (`count` per kategori) akurat — di `resume_chunks` angka itu over-count ~16x.

Tiap resume diringkas LLM jadi satu baris **profil padat berurutan tetap**:
`<role> · <senioritas/tahun jika eksplisit> · <domain> · <3–5 skill inti>` (Bahasa Inggris). Guardrail anti-halusinasi hidup di prompt (titik generasi) — Python pasca-proses tidak bisa menilai apakah suatu klaim ter-grounding di resume.

Pola **resume-safe identik** dengan ingestion chunk: point ID deterministik `uuid5(INGESTION_NAMESPACE, f"summary_{resume_id}")`, satu `upsert()` atomic per resume, scroll-skip yang sudah jadi. Re-run aman & idempoten.

Hanya LLM call summary yang di-instrumentasi Langfuse (client chat terpisah) — embedding tetap tidak di-trace.

In [26]:
from langfuse import Langfuse, get_client
from langfuse.openai import OpenAI as LangfuseOpenAI

# Instrument ONLY the summary LLM call. embed_texts keeps the plain
# openai_client -> embeddings stay untraced (locked Langfuse scope).
Langfuse(
    public_key=config.LANGFUSE_PUBLIC_KEY,
    secret_key=config.LANGFUSE_SECRET_KEY,
    host=config.LANGFUSE_HOST,
)
langfuse_client = get_client()
chat_client = LangfuseOpenAI(api_key=config.OPENAI_API_KEY)

SUMMARY_SYSTEM_PROMPT = """You generate ultra-compact candidate triage profiles for a recruitment
search engine. Given one resume, output EXACTLY ONE line — no preamble,
no quotes, no markdown, no trailing notes.

Fixed format (fields in this order, joined by " · "):
<primary role> · <seniority + total years> · <domain/industry> · <3-5 core skills>

Rules:
- English. Telegraphic phrases, not sentences.
- <primary role>: the candidate's most senior or most recent job title, as written.
- <seniority + total years>: include ONLY when explicitly evidenced (title
  contains Senior/Lead/Manager/Director, or a total years figure is stated).
  If there is no explicit signal, OMIT this whole field AND its separator —
  never write "unknown", never guess.
- <domain/industry>: the industry the candidate works in, from explicit
  employer/role context only.
- <3-5 core skills>: 3 to 5 most prominent skills/tools explicitly present,
  comma-separated. Never invent a skill not in the resume.
- Use only information in the resume. Do not fabricate anything.
- Output the single profile line and nothing else.

Examples:
Senior Data Engineer · Senior, 7 yrs · fintech / payments · Python, Spark, AWS, Airflow, data modeling
Graphic Designer · retail & e-commerce · Adobe Photoshop, Illustrator, branding, layout design"""


def summarize_one(resume_text):
    """One compact triage profile via gpt-4o-mini. Hand-rolled backoff on
    transient errors (same policy as embed_texts). Fail-loud after 5 tries —
    resume-safe loop makes a hard stop recoverable. Langfuse drop-in
    auto-traces model/io/token/cost for this call only."""
    for attempt in range(5):
        try:
            resp = chat_client.chat.completions.create(
                model=config.CHAT_MODEL,
                temperature=0.2,
                messages=[
                    {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
                    {"role": "user", "content": resume_text},
                ],
            )
            return resp.choices[0].message.content.strip()
        except _RETRYABLE as e:
            if attempt == 4:
                raise
            wait = 2 ** attempt
            print(f"  transient LLM error ({type(e).__name__}); "
                  f"retry {attempt + 1}/4 in {wait}s")
            time.sleep(wait)


print("summarize_one() ready — Langfuse-instrumented, embeddings untraced")

summarize_one() ready — Langfuse-instrumented, embeddings untraced


In [27]:
from qdrant_client.models import PointStruct  # cell 17 skipped -> import here

SUMMARY_COL = config.SUMMARY_COLLECTION

# Resume-from-crash: skip resumes whose summary is already in Qdrant.
done = scroll_done_resume_ids(SUMMARY_COL)
pending = df.loc[~df["ID"].isin(done), ["ID", "Category", "Resume_clean"]]
print(f"Resumes total : {len(df):,}")
print(f"Already done  : {len(done):,}")
print(f"Pending       : {len(pending):,}")

t0 = time.time()
processed = 0
for r in pending.itertuples(index=False):
    rid, category, text = r.ID, r.Category, r.Resume_clean
    summary = summarize_one(text)
    vector = embed_texts([summary])[0]
    pid = str(uuid.uuid5(INGESTION_NAMESPACE, f"summary_{rid}"))
    # One upsert call per resume -> Qdrant atomic per request.
    qdrant.upsert(
        collection_name=SUMMARY_COL,
        points=[PointStruct(
            id=pid,
            vector=vector,
            payload={
                "page_content": summary,
                "metadata": {"resume_id": rid, "category": category},
            },
        )],
    )
    processed += 1
    if processed % 50 == 0:
        el = time.time() - t0
        rate = processed / el if el else 0
        eta = (len(pending) - processed) / rate if rate else 0
        print(f"  {processed:,}/{len(pending):,} | {el:,.0f}s | ~{eta:,.0f}s left")

langfuse_client.flush()  # force async-batched traces out before kernel idles
print(f"Summaries ingested: {processed:,} resumes in {time.time() - t0:,.0f}s.")

Resumes total : 2,481
Already done  : 0
Pending       : 2,481
  50/2,481 | 91s | ~4,430s left
  100/2,481 | 185s | ~4,403s left
  150/2,481 | 272s | ~4,226s left
  200/2,481 | 359s | ~4,099s left
  250/2,481 | 447s | ~3,990s left
  300/2,481 | 532s | ~3,868s left
  350/2,481 | 620s | ~3,773s left
  400/2,481 | 704s | ~3,663s left
  450/2,481 | 786s | ~3,548s left


Failed to export span batch code: None, reason: HTTPSConnectionPool(host='cloud.langfuse.com', port=443): Read timed out. (read timeout=4.999988079071045)


  500/2,481 | 1,474s | ~5,841s left
  550/2,481 | 1,561s | ~5,479s left
  600/2,481 | 1,647s | ~5,162s left
  650/2,481 | 1,734s | ~4,884s left
  700/2,481 | 1,819s | ~4,628s left
  750/2,481 | 1,911s | ~4,410s left
  800/2,481 | 2,001s | ~4,206s left
  850/2,481 | 2,089s | ~4,009s left
  900/2,481 | 2,175s | ~3,822s left
  950/2,481 | 2,263s | ~3,647s left
  1,000/2,481 | 2,351s | ~3,481s left
  1,050/2,481 | 2,435s | ~3,318s left
  1,100/2,481 | 2,520s | ~3,163s left
  1,150/2,481 | 2,605s | ~3,015s left
  1,200/2,481 | 2,685s | ~2,867s left
  1,250/2,481 | 2,769s | ~2,727s left
  1,300/2,481 | 2,854s | ~2,593s left
  1,350/2,481 | 2,941s | ~2,464s left
  1,400/2,481 | 3,024s | ~2,335s left
  1,450/2,481 | 3,118s | ~2,217s left
  1,500/2,481 | 3,203s | ~2,095s left
  1,550/2,481 | 3,286s | ~1,974s left
  1,600/2,481 | 3,373s | ~1,857s left
  1,650/2,481 | 3,464s | ~1,744s left
  1,700/2,481 | 3,548s | ~1,630s left
  1,750/2,481 | 3,639s | ~1,520s left
  1,800/2,481 | 3,750s | ~1,419s

## 8. Validasi ingestion

Tiga gate empiris sebelum data layer dianggap selesai:

- **8a — kelengkapan:** `points_count` & jumlah `resume_id` unik harus = jumlah resume bersih. Membuktikan "selesai" dengan angka, bukan asumsi. Kalau kurang → re-run cell 7b (resume-safe).
- **8b — sanity format:** eyeball summary acak. Format `<role> · <senioritas?> · <domain> · <skills>` konsisten? Aturan omission terlihat bekerja (sebagian ada field senioritas, sebagian tidak)? Bahasa Inggris, tanpa preamble?
- **8c — linchpin arsitektur:** filter di key bersarang `metadata.category` benar-benar bekerja (kalau index salah/flat → 0 hasil atau error). Sekaligus bukti properti agregasi: kardinalitas 1 point : 1 kandidat → `count` per kategori cocok persis dengan ground truth dataframe.

In [34]:
si = qdrant.get_collection(config.SUMMARY_COLLECTION)
done_sum = scroll_done_resume_ids(config.SUMMARY_COLLECTION)
target = len(df)

print(f"points_count   : {si.points_count:,}  (target {target:,})")
print(f"resume_id uniq : {len(done_sum):,}  (target {target:,})")

missing = set(df["ID"]) - done_sum
if missing:
    print(f"MISSING        : {len(missing):,} -> re-run cell 7b (resume-safe). "
          f"sample {list(missing)[:10]}")
else:
    print("MISSING        : 0  -> complete")

points_count   : 2,481  (target 2,481)
resume_id uniq : 2,481  (target 2,481)
MISSING        : 0  -> complete


In [35]:
import random

pts, _ = qdrant.scroll(
    config.SUMMARY_COLLECTION, with_payload=True, with_vectors=False, limit=10_000
)
rng = random.Random(42)
sample = rng.sample(pts, min(8, len(pts)))

print(f"Sanity: {len(sample)} random summaries of {len(pts):,}\n")
for p in sample:
    m = p.payload["metadata"]
    print(f"id={m['resume_id']} | {m['category']}")
    print(f"  {p.payload['page_content']}")
    print()

Sanity: 8 random summaries of 2,481

id=32140087 | SALES
  Marketing Project Production Manager · Senior, 8 yrs · retail & e-commerce · Project Management, Print Production, Quality Control, Team Building, Adobe Creative Suite

id=85766635 | PUBLIC-RELATIONS
  Human Resources Administrative Assistant · 1 yr · human resources · Microsoft Office, HRIS, customer service, event coordination, social media management

id=14364597 | PUBLIC-RELATIONS
  Public Relations & Development Associate · 3 yrs · nonprofit · event planning, client relations, team management, customer service, communication skills

id=26767199 | FINANCE
  Finance Manager · 15 yrs · finance · GAAP, SOX compliance, financial reporting, analytical skills, team management

id=18365443 | HEALTHCARE
  Healthcare Analyst · 26 yrs · healthcare · Revenue Cycle Management, SQL, financial consulting, process improvement, customer service

id=27662298 | CHEF
  Executive Sous Chef · 25 yrs · hospitality · staff development, inventory 

In [36]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

test_cat = "INFORMATION-TECHNOLOGY"
qvec = embed_texts(["senior software engineer with cloud experience"])[0]

res = qdrant.query_points(
    collection_name=config.SUMMARY_COLLECTION,
    query=qvec,
    query_filter=Filter(must=[
        FieldCondition(key="metadata.category", match=MatchValue(value=test_cat))
    ]),
    limit=5,
    with_payload=True,
)

print(f"Hard-filter test (category={test_cat}):")
for p in res.points:
    m = p.payload["metadata"]
    print(f"  score={p.score:.3f} | {m['category']} | id={m['resume_id']}")
    print(f"    {p.payload['page_content'][:120]}")
ok = all(p.payload["metadata"]["category"] == test_cat for p in res.points)
print(f"  ALL match filter: {ok}  (must be True -> metadata.category index works)")

print("\nPer-category count (Qdrant count vs df ground truth):")
for cat in ["HR", "INFORMATION-TECHNOLOGY", "BPO", "AUTOMOBILE"]:
    qd = qdrant.count(
        config.SUMMARY_COLLECTION,
        count_filter=Filter(must=[
            FieldCondition(key="metadata.category", match=MatchValue(value=cat))
        ]),
        exact=True,
    ).count
    gt = int((df["Category"] == cat).sum())
    flag = "OK" if qd == gt else "MISMATCH"
    print(f"  {cat:24s} qdrant={qd:4d}  df={gt:4d}  [{flag}]")

Hard-filter test (category=INFORMATION-TECHNOLOGY):
  score=0.593 | INFORMATION-TECHNOLOGY | id=32959732
    Senior Director, Information Technology · Senior, 1 yr · education · Cloud technologies, ERP implementation, project man
  score=0.565 | INFORMATION-TECHNOLOGY | id=21283365
    Director of Information Technology · Senior, 18 yrs · IT services · Cloud Environments, Cyber Security, Project Manageme
  score=0.564 | INFORMATION-TECHNOLOGY | id=33381211
    Vice President Information Technology - Software Engineering · 20+ yrs · technology · Software Engineering, Strategic Pl
  score=0.561 | INFORMATION-TECHNOLOGY | id=17681064
    Information Technology Senior Manager · Senior, 15+ yrs · health & wellness · Vendor management, project management, neg
  score=0.550 | INFORMATION-TECHNOLOGY | id=83816738
    Java Full Stack Developer · 2 yrs · software development · Java, Spring, Microservices, AngularJS, Selenium
  ALL match filter: True  (must be True -> metadata.category index work

In [37]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

# 8d — chunk-filter parity. 8c proved resume_summaries; resume_chunks index
# was ALSO just created (was missing all along), so its category filter is
# unverified. Don't assume it works — measure, before closing the data layer.
test_cat_c = "HEALTHCARE"
qvec_c = embed_texts(["registered nurse with ICU experience"])[0]

res_c = qdrant.query_points(
    collection_name=config.CHUNK_COLLECTION,
    query=qvec_c,
    query_filter=Filter(must=[
        FieldCondition(key="metadata.category", match=MatchValue(value=test_cat_c))
    ]),
    limit=5,
    with_payload=True,
)

print(f"Chunk hard-filter test (category={test_cat_c}):")
for p in res_c.points:
    m = p.payload["metadata"]
    print(f"  score={p.score:.3f} | {m['category']} | id={m['resume_id']} | sec={m['section']}")
    print(f"    {p.payload['page_content'][:110]}")
ok_c = all(p.payload["metadata"]["category"] == test_cat_c for p in res_c.points)
print(f"  ALL match filter: {ok_c}  (must be True -> resume_chunks index works)")

Chunk hard-filter test (category=HEALTHCARE):
  score=0.584 | HEALTHCARE | id=17960690 | sec=experience
      State      Practiced as a Registered Nurse in the Neonatal Intensive Care Unit
  score=0.557 | HEALTHCARE | id=16356151 | sec=summary
    Dedicated RN with over 20 years experience in nursing seeking career transition into a new clinical setting. A
  score=0.557 | HEALTHCARE | id=14062078 | sec=title
    REGISTERED NURSE
  score=0.557 | HEALTHCARE | id=46349752 | sec=title
    REGISTERED NURSE
  score=0.552 | HEALTHCARE | id=31395710 | sec=summary
    Licensed Practical Nurse with 15 years in providing direct care under RN and MD supervision in diagnosis, trea
  ALL match filter: True  (must be True -> resume_chunks index works)


## ⚠️ Reset destruktif (opsional, di akhir notebook)

Hapus kedua collection. **Hanya** untuk full reset — mis. ganti strategi chunking, atau memperbaiki index payload. Di-guard oleh `RUN_DESTRUCTIVE` agar "Run All" tidak memicunya tanpa sengaja. Setelah dijalankan, **re-run cell `ensure_collection`** untuk recreate (kini dengan index `metadata.category`).

In [ ]:
RUN_DESTRUCTIVE = False  # flip True HANYA untuk wipe total

if RUN_DESTRUCTIVE:
    for _c in (config.CHUNK_COLLECTION, config.SUMMARY_COLLECTION):
        qdrant.delete_collection(_c)
        print(f"deleted {_c}")
    print("Sekarang re-run cell ensure_collection untuk recreate (index metadata.category).")
else:
    print("RUN_DESTRUCTIVE = False -> no-op. Flip True untuk wipe collections.")